In [3]:
print(123)

123


In [4]:
import pandas as pd

df = pd.read_csv('dataset/coffee_analysis.csv')
df.head()

,name,roaster,roast,loc_country,origin_1,origin_2,100g_USD,rating,review_date,desc_1,desc_2,desc_3
0,“Sweety” Espresso Blend,A.R.C.,Medium-Light,Hong Kong,Panama,Ethiopia,14.32,95,November 2017,"Evaluated as espresso. Sweet-toned, deeply ric...",An espresso blend comprised of coffees from Pa...,A radiant espresso blend that shines equally i...
1,Flora Blend Espresso,A.R.C.,Medium-Light,Hong Kong,Africa,Asia Pacific,9.05,94,November 2017,"Evaluated as espresso. Sweetly tart, floral-to...",An espresso blend comprised of coffees from Af...,"A floral-driven straight shot, amplified with ..."
2,Ethiopia Shakiso Mormora,Revel Coffee,Medium-Light,United States,Guji Zone,Southern Ethiopia,4.70,92,November 2017,"Crisply sweet, cocoa-toned. Lemon blossom, roa...",This coffee tied for the third-highest rating ...,"A gently spice-toned, floral- driven wet-proce..."
3,Ethiopia Suke Quto,Roast House,Medium-Light,United States,Guji Zone,Oromia Region,4.19,92,November 2017,"Delicate, sweetly spice-toned. Pink peppercorn...",This coffee tied for the third-highest rating ...,Lavender-like flowers and hints of zesty pink ...
4,Ethiopia Gedeb Halo Beriti,Big Creek Coffee Roasters,Medium,United States,Gedeb District,Gedeo Zone,4.85,94,November 2017,"Deeply sweet, subtly pungent. Honey, pear, tan...",Southern Ethiopia coffees like this one are pr...,A deeply and generously lush cup saved from se...


In [5]:
cols = df.columns.tolist()
type(cols)

list

In [6]:
cols

['name',
 'roaster',
 'roast',
 'loc_country',
 'origin_1',
 'origin_2',
 '100g_USD',
 'rating',
 'review_date',
 'desc_1',
 'desc_2',
 'desc_3']

In [7]:
from minsearch import Index

index = Index(
    text_fields = cols
)


In [8]:
df.dropna(inplace=True)

In [9]:
df.isna().sum()

name           0
roaster        0
roast          0
loc_country    0
origin_1       0
origin_2       0
100g_USD       0
rating         0
review_date    0
desc_1         0
desc_2         0
desc_3         0
dtype: int64

In [10]:
df.dtypes

name               str
roaster            str
roast              str
loc_country        str
origin_1           str
origin_2           str
100g_USD       float64
rating           int64
review_date        str
desc_1             str
desc_2             str
desc_3             str
dtype: object

In [11]:
df[['100g_USD', 'rating']] = df[['100g_USD', 'rating']].astype(str)

In [12]:
docs = df.to_dict(orient='records')

In [13]:
index.fit(docs)

In [14]:
boost_dict = {'desc_1': 2, 'desc_2': 1.5, 'desc_3': 1.5}

q = "coffe bean from indonesia"

results = index.search(query=q, num_results=5, boost_dict=boost_dict)

In [15]:
results

[{'name': 'Indonesia Flores Island',
  'roaster': 'Green Stone Coffee',
  'roast': 'Light',
  'loc_country': 'Taiwan',
  'origin_1': 'Flores Island',
  'origin_2': 'Indonesia',
  '100g_USD': '7.11',
  'rating': '92',
  'review_date': 'April 2018',
  'desc_1': 'Crisp, richly sweet-savory. Maple syrup, gardenia-like flowers, fresh humus, pomegranate, baker’s chocolate in aroma and cup. Deeply sweet in structure, with savory undertones; delicate, silky mouthfeel. The finish is driven by gardenia-like florals and sweet earth tones in the short, baker’s chocolate and tart fruit in the long.',
  'desc_2': 'Produced by Lambertus Siba Hurek entirely of the Typica variety of Arabica and processed by the Indonesia wet-hulled method. Flores is an island in Indonesia, part of a chain of islands that extends east of Java and Bali. With a recent focus on quality and assistance from international aid groups, Flores is beginning to create a name for itself in specialty coffee circles. Green Stone Coff

In [16]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

In [17]:
query = "What is the best coffee bean from indonesia?"
context = index.search(query=query, num_results=5, boost_dict=boost_dict)

In [53]:
INSTRUCTIONS = """
# Role and Objective
You are an expert Q-Grader and sensory analyst specializing in coffee evaluation. Your core objective is to analyze coffee beans by combining the user's query with the provided context from the coffee database (RAG context). Deliver precise, objective reviews that focus on flavor profiles, extraction advice, and quality metrics.
# Grounding Constraints (Anti-Hallucination)
1. Rely strictly on the information provided within the context blocks. 
2. Do not extrapolate, invent roasters, or assume flavor profiles not explicitly detailed in the source material.
3. If the context contains insufficient data to answer the query accurately, state: "Insufficient information in the current database to review this coffee."
4. If there is a contradiction in the text (e.g., conflicting optimal temperatures), present both options with their respective sources.

# Structured Evaluation Framework
When generating a review, extract and format the data using the following taxonomy:

1. **Origin Profile:** Region, farm/estate, processing method (natural, washed, honey, anaerobic), and roast level.
2. **Sensory Analysis:** Split into explicit categories:
   - *Aroma:* Dry fragrance and wet aroma notes.
   - *Flavor & Acidity:* Primary tasting notes, type of acidity (e.g., citric, malic, phosphoric), and intensity.
   - *Body & Finish:* Mouthfeel texture (e.g., juicy, tea-like, syrupy) and aftertaste persistence.
   
# Tone and Style Guide
- **Tone:** Authoritative, descriptive, and precise. Avoid generic marketing hype ("amazing," "mind-blowing"). Use technical coffee vernacular (e.g., "bright phosphoric acidity," "silky mouthfeel").
- **Formatting:** Use structured headers, bold parameters, and concise bullet points.
- **Brevity:** Eliminate conversational filler, introductory remarks, and concluding summaries. Begin immediately with the structured evaluation.

# Input Format Structure
The inputs will be provided as follows:
{context}: Retreived snippets from the database.
{query}: The user's specific request.
""".strip()

In [34]:
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

In [ ]:
context_chunks = []

for idx, result in enumerate(results, start=1):
    # Construct a single, highly structured semantic block for each coffee
    chunk = (
        f"--- Document {idx} ---\n"
        f"Coffee Name: {result['name']}\n"
        f"Origin: {result['origin_1']}\n"
        f"Sensory Notes (desc_1): {result['desc_1']}\n"
        f"Flavor Profile (desc_2): {result['desc_2']}\n"
        f"Extraction Advice (desc_3): {result['desc_3']}\n"
        f"Roast Level: {result['roast']}\n"
        f"Quality Rating: {result['rating']}"
    )
    context_chunks.append(chunk)

# Combine all isolated blocks into one unified context string for the user prompt
context_string = "\n\n".join(context_chunks)

In [65]:
PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()

PROMPT = PROMPT_TEMPLATE.format(
    question=query,
    context=context_string
)

In [67]:
client = OpenAI()

In [68]:
input_messages = [
        {'role': 'developer', 'content': INSTRUCTIONS},
        {'role': 'user', 'content': PROMPT}
    ]

response = client.responses.create(
    model="gpt-5.4-mini",
    input=input_messages
)

In [69]:
print(response.output_text)

**Best Indonesia coffee bean in the current database: Indonesia Flores Island**

**Origin Profile**
- **Region:** Flores Island, Indonesia
- **Farm/Producer:** Lambertus Siba Hurek
- **Process:** Indonesia wet-hulled
- **Variety:** 100% Typica Arabica
- **Roast Level:** Light

**Sensory Analysis**
- **Aroma:** Maple syrup, gardenia-like flowers, fresh humus, pomegranate, baker’s chocolate
- **Flavor & Acidity:** Deeply sweet structure with savory undertones; tart fruit is explicitly present in the cup and finish. The profile is crisp, richly sweet-savory, with floral and sweet earth tones
- **Body & Finish:** Delicate, silky mouthfeel; finish is short with gardenia-like florals and sweet earth notes, extending into a longer baker’s chocolate and tart-fruit impression

**Quality Metric**
- **Quality Rating:** 92

**Why this is the best Indonesia bean here**
- Among the coffees whose origin is Indonesia or include Indonesia as a primary component, this is the only single-origin Indonesia

In [70]:
response.usage.input_tokens
response.usage.output_tokens

290

In [71]:
usage = response.usage

input_price_per_million = 0.75
output_price_per_million = 4.50

input_cost = (usage.input_tokens / 1_000_000) * input_price_per_million
output_cost = (usage.output_tokens / 1_000_000) * output_price_per_million
total_cost = input_cost + output_cost
print(total_cost)

0.00251175


In [1]:
from ingest_data import load_data, build_index

documents = load_data()
index = build_index(documents)

In [2]:
from rag_app import RAGBase
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI

assistant = RAGBase(index=index, llm_client=OpenAI())

In [ ]:
query = "good coffee bean for making espresso"

response = assistant.rag(query=query)
print(response.output_text)

**Best espresso candidate from the database: Brazil Conquista**

**Origin Profile**
- **Region:** Bahia
- **Farm/Estate:** Conquista Farm
- **Process:** Pulped natural
- **Roast Level:** Medium-Light

**Sensory Analysis**
- **Aroma:** Baker’s chocolate, dried cherry, magnolia-like flowers, cedar, hazelnut
- **Flavor & Acidity:** Sweetly pungent, chocolate-toned; sweet structure with **gentle, rounded acidity**
- **Body & Finish:** **Velvety mouthfeel**; finish carries magnolia and cherry short, then cedar and hazelnut with gentle drying; good chocolate throughout

**Why it suits espresso**
- The **chocolate-forward profile**, **rounded acidity**, and **velvety body** are structurally well-suited to espresso extraction.
- The pulped natural process can support a sweeter, more textured cup in espresso.

**Other viable option**
- **Bright House Signature Coffee**  
  - Richly sweet, chocolaty with **brisk acidity** and **full, creamy mouthfeel**
  - If you want a brighter, more citrus-acc